In [ ]:
import pandas as pd
import json
import ast
import random
import asyncio
import os
import re
from tqdm import tqdm
from openai import AsyncOpenAI

# -----------------------------
# CONFIG
# -----------------------------
INPUT_FILE = "input.csv"
OUTPUT_FILE = "output_narrative_probes.csv"

BATCH_SIZE = 10
CONCURRENT_REQUESTS = 5
MAX_RETRIES = 3

OPENAI_MODEL = os.getenv("OPENAI_MODEL", "gpt-5.4-nano")

random.seed(4000)


client = AsyncOpenAI(api_key="sk-...")

ALT_CONDITIONS = [
    "group alep", "group graeca", "context alpha",
    "setting dalet", "population delta", "environment theta",
    "group waw", "context zayin", "sample yod", "sample lamed"
]

VALID_LABELS = {
    "Supported",
    "Not Supported",
    "Not enough evidence"
}

EXPECTED_PROBES = {
    1: {
        "probe_type": "relation_contradicted",
        "gold_label": "Not Supported"
    },
    2: {
        "probe_type": "condition_replaced",
        "gold_label": "Not enough evidence"
    },
    3: {
        "probe_type": "condition_removed",
        "gold_label": "Not enough evidence"
    },
    4: {
        "probe_type": "supported",
        "gold_label": "Supported"
    },
    5: {
        "probe_type": "condition_contradicted",
        "gold_label": "Not Supported"
    }
}

# -----------------------------
# HELPERS
# -----------------------------
def parse_tuple_value(x):
    if isinstance(x, dict):
        return x

    try:
        return ast.literal_eval(x)
    except Exception:
        return json.loads(x)


def extract_json(text):
    text = text.strip()

    text = re.sub(r"^```json", "", text, flags=re.IGNORECASE).strip()
    text = re.sub(r"^```", "", text).strip()
    text = re.sub(r"```$", "", text).strip()

    try:
        return json.loads(text)
    except Exception:
        pass

    start = text.find("[")
    end = text.rfind("]")

    if start != -1 and end != -1 and end > start:
        return json.loads(text[start:end + 1])

    raise ValueError("Could not parse JSON from model response.")


def validate_probe(probe, expected_probe_number, e1, e2):
    required_keys = [
        "probe_number",
        "probe_type",
        "evidence",
        "claim",
        "answer"
    ]

    for key in required_keys:
        if key not in probe:
            return False, f"Missing key: {key}"

    probe_number = probe["probe_number"]
    probe_type = str(probe["probe_type"]).strip()
    evidence = str(probe["evidence"]).strip()
    claim = str(probe["claim"]).strip()
    answer = str(probe["answer"]).strip()

    if probe_number != expected_probe_number:
        return False, f"Expected probe_number {expected_probe_number}, got {probe_number}"

    expected_probe_type = EXPECTED_PROBES[expected_probe_number]["probe_type"]
    expected_gold_label = EXPECTED_PROBES[expected_probe_number]["gold_label"]

    if probe_type != expected_probe_type:
        return False, f"Expected probe_type {expected_probe_type}, got {probe_type}"

    if answer != expected_gold_label:
        return False, f"Expected answer {expected_gold_label}, got {answer}"

    if answer not in VALID_LABELS:
        return False, f"Invalid answer label: {answer}"

    if not evidence:
        return False, "Evidence is empty"

    if not claim:
        return False, "Claim is empty"

    if "?" in claim:
        return False, "Claim should be a statement, not a question"

    if e1 not in claim:
        return False, "Claim does not contain e1 exactly"

    if e2 not in claim:
        return False, "Claim does not contain e2 exactly"

    if len(evidence.split()) < 25:
        return False, "Evidence is too short"

    if len(evidence.split()) > 120:
        return False, "Evidence is too long"

    return True, "Valid"


# -----------------------------
# PROMPT
# -----------------------------
def build_batch_prompt(batch_rows):
    items = []

    for i, row in enumerate(batch_rows):
        tup = parse_tuple_value(row["Tuple"])
        alt_condition = random.choice(ALT_CONDITIONS)

        items.append(f"""
{i + 1}.
tuple_id: {row["Tuple ID"]}
e1: {tup["e1"]}
e2: {tup["e2"]}
relation: {tup["relation"]}
condition: {tup["condition"]}
alt_condition: {alt_condition}
""")

    return f"""
You are generating narrative classification probes from conditional relation tuples.

For EACH item, create exactly 5 probes.

Each probe must contain:
- probe_number
- probe_type
- evidence
- claim
- answer

The answer must be exactly one of:
- Supported
- Not Supported
- Not enough evidence

The evidence should be narrative-wise, not a repeated template.
Use styles such as:
- research memo
- clinical note
- lab summary
- field observation
- analyst paragraph
- abstract-style fragment
- technical report note

General rules:
- Keep e1 and e2 EXACTLY as written.
- Do not write yes/no questions.
- The claim must be a statement, not a question.
- The claim must contain e1 and e2 exactly.
- Avoid repetitive wording across probes.
- Avoid starting claims with "Does", "Is it true", "Is it false", or "Can we say".
- Evidence should usually be 35 to 90 words.
- Do not explain the answer.
- Return only valid JSON.
- Do not include markdown.

CRITICAL PROBE DEFINITIONS:

Probe 1:
- probe_number: 1
- probe_type: "relation_contradicted"
- Meaning: The claim must contradict the original relation.
- Keep the original condition.
- This is a RELATION contradiction, not a condition contradiction.
- answer: "Not Supported"

Probe 2:
- probe_number: 2
- probe_type: "condition_replaced"
- Meaning: The claim must replace the original condition with alt_condition.
- Do not contradict the relation.
- The issue is that the condition has been swapped.
- answer: "Not enough evidence"

Probe 3:
- probe_number: 3
- probe_type: "condition_removed"
- Meaning: The claim must omit the condition entirely.
- Do not mention the original condition.
- Do not mention alt_condition.
- The issue is that the relation is now stated without its required condition.
- answer: "Not enough evidence"

Probe 4:
- probe_number: 4
- probe_type: "supported"
- Meaning: The claim must preserve the original e1, relation, e2, and condition.
- answer: "Supported"

Probe 5:
- probe_number: 5
- probe_type: "condition_contradicted"
- Meaning: The claim must contradict or oppose the original condition.
- Do not contradict the relation itself.
- This is a CONDITION contradiction, not a relation contradiction.
- answer: "Not Supported"

For all 5 probes:
- The evidence should be written as a natural paragraph.
- The evidence should give the reader enough context to judge the claim.
- The claim should be one sentence.
- The claim should be classification-ready.

Return this exact JSON structure:

[
  {{
    "id": 1,
    "tuple_id": "...",
    "probes": [
      {{
        "probe_number": 1,
        "probe_type": "relation_contradicted",
        "evidence": "...",
        "claim": "...",
        "answer": "Not Supported"
      }},
      {{
        "probe_number": 2,
        "probe_type": "condition_replaced",
        "evidence": "...",
        "claim": "...",
        "answer": "Not enough evidence"
      }},
      {{
        "probe_number": 3,
        "probe_type": "condition_removed",
        "evidence": "...",
        "claim": "...",
        "answer": "Not enough evidence"
      }},
      {{
        "probe_number": 4,
        "probe_type": "supported",
        "evidence": "...",
        "claim": "...",
        "answer": "Supported"
      }},
      {{
        "probe_number": 5,
        "probe_type": "condition_contradicted",
        "evidence": "...",
        "claim": "...",
        "answer": "Not Supported"
      }}
    ]
  }}
]

Items:
{''.join(items)}
"""


# -----------------------------
# GENERATE ONE BATCH
# -----------------------------
async def process_batch(batch_df):
    batch_records = batch_df.to_dict("records")

    for attempt in range(1, MAX_RETRIES + 1):
        try:
            prompt = build_batch_prompt(batch_records)

            res = await client.chat.completions.create(
                model=OPENAI_MODEL,
                messages=[
                    {
                        "role": "system",
                        "content": "You generate valid JSON only. No markdown. No explanations."
                    },
                    {
                        "role": "user",
                        "content": prompt
                    }
                ],
                temperature=0.9
            )

            content = res.choices[0].message.content
            outputs = extract_json(content)

            if not isinstance(outputs, list):
                raise ValueError("Model output is not a JSON list.")

            results = []

            for i, row in enumerate(batch_records):
                tup = parse_tuple_value(row["Tuple"])

                if i >= len(outputs):
                    results.append({
                        "Tuple ID": row["Tuple ID"],
                        "Tuple": row["Tuple"],
                        "e1": tup.get("e1", ""),
                        "e2": tup.get("e2", ""),
                        "relation": tup.get("relation", ""),
                        "condition": tup.get("condition", ""),
                        "Probe Number": "",
                        "Probe Type": "",
                        "Evidence": "",
                        "Claim": "",
                        "Gold Label": "",
                        "Validation Status": "Invalid",
                        "Validation Note": "Missing output for this row"
                    })
                    continue

                out = outputs[i]
                probes = out.get("probes", [])

                if not isinstance(probes, list) or len(probes) != 5:
                    results.append({
                        "Tuple ID": row["Tuple ID"],
                        "Tuple": row["Tuple"],
                        "e1": tup.get("e1", ""),
                        "e2": tup.get("e2", ""),
                        "relation": tup.get("relation", ""),
                        "condition": tup.get("condition", ""),
                        "Probe Number": "",
                        "Probe Type": "",
                        "Evidence": "",
                        "Claim": "",
                        "Gold Label": "",
                        "Validation Status": "Invalid",
                        "Validation Note": "Expected exactly 5 probes"
                    })
                    continue

                probes = sorted(probes, key=lambda x: x.get("probe_number", 999))

                for expected_probe_number, probe in enumerate(probes, start=1):
                    is_valid, note = validate_probe(
                        probe=probe,
                        expected_probe_number=expected_probe_number,
                        e1=tup["e1"],
                        e2=tup["e2"]
                    )

                    results.append({
                        "Tuple ID": row["Tuple ID"],
                        "Tuple": row["Tuple"],
                        "e1": tup.get("e1", ""),
                        "e2": tup.get("e2", ""),
                        "relation": tup.get("relation", ""),
                        "condition": tup.get("condition", ""),
                        "Probe Number": probe.get("probe_number", ""),
                        "Probe Type": probe.get("probe_type", ""),
                        "Evidence": probe.get("evidence", ""),
                        "Claim": probe.get("claim", ""),
                        "Gold Label": probe.get("answer", ""),
                        "Validation Status": "Valid" if is_valid else "Invalid",
                        "Validation Note": note
                    })

            return results

        except Exception as e:
            print(f"Batch failed on attempt {attempt}/{MAX_RETRIES}: {e}")

    failed_results = []

    for _, row in batch_df.iterrows():
        tup = parse_tuple_value(row["Tuple"])

        failed_results.append({
            "Tuple ID": row["Tuple ID"],
            "Tuple": row["Tuple"],
            "e1": tup.get("e1", ""),
            "e2": tup.get("e2", ""),
            "relation": tup.get("relation", ""),
            "condition": tup.get("condition", ""),
            "Probe Number": "",
            "Probe Type": "",
            "Evidence": "",
            "Claim": "",
            "Gold Label": "",
            "Validation Status": "Failed",
            "Validation Note": "Batch failed after all retries"
        })

    return failed_results


# -----------------------------
# MAIN ASYNC RUNNER
# -----------------------------
async def main():
    df = pd.read_csv(INPUT_FILE)

    required_columns = {"Tuple ID", "Tuple"}
    missing_columns = required_columns - set(df.columns)

    if missing_columns:
        raise ValueError(f"Missing required columns: {missing_columns}")

    if os.path.exists(OUTPUT_FILE):
        existing = pd.read_csv(OUTPUT_FILE)

        if "Tuple ID" in existing.columns and "Evidence" in existing.columns:
            completed_ids = set(
                existing[
                    existing["Evidence"].notna()
                    & (existing["Evidence"].astype(str).str.strip() != "")
                ]["Tuple ID"].unique()
            )

            df = df[~df["Tuple ID"].isin(completed_ids)]
            results = existing.to_dict("records")
        else:
            results = []
    else:
        results = []

    if df.empty:
        print("No new rows to process.")
        print(f"Existing output file: {OUTPUT_FILE}")
        return

    batches = [
        df.iloc[i:i + BATCH_SIZE]
        for i in range(0, len(df), BATCH_SIZE)
    ]

    semaphore = asyncio.Semaphore(CONCURRENT_REQUESTS)

    async def sem_task(batch):
        async with semaphore:
            return await process_batch(batch)

    tasks = [sem_task(batch) for batch in batches]

    for future in tqdm(asyncio.as_completed(tasks), total=len(tasks)):
        batch_results = await future

        results.extend(batch_results)

        pd.DataFrame(results).to_csv(OUTPUT_FILE, index=False)

    print("Done")
    print(f"Saved output to: {OUTPUT_FILE}")


# -----------------------------
# RUN IN JUPYTER / COLAB
# -----------------------------
await main()